<a href="https://colab.research.google.com/github/avikumart/LLM-GenAI-Transformers-Notebooks/blob/main/AgenticAIDeepLearningAI/Research_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearchResults
from langchain_experimental.utilities import PythonREPL
from langchain_core.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. SET UP OBSERVABILITY (LangSmith)
# These environment variables enable automatic tracing to LangSmith.
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Research-Agent-v2026"
# Note: Ensure LANGCHAIN_API_KEY, OPENAI_API_KEY, and TAVILY_API_KEY are set in your environment.

# 2. DEFINE CUSTOM TOOLS

@tool
def python_repl_tool(code: str):
    """
    Executes Python code in a local REPL environment.
    Use this for math, data analysis, or generating charts.
    """
    try:
        repl = PythonREPL()
        result = repl.run(code)
        return f"Successfully executed. Output:\n{result}"
    except Exception as e:
        return f"Failed to execute code. Error: {str(e)}"

@tool
def save_draft(content: str, filename: str = "research_draft.txt"):
    """
    Saves a research draft or final report to a local text file.
    Use this when you have finished gathering information and are ready to output the final result.
    """
    try:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_name = f"{timestamp}_{filename}"

        with open(final_name, "w", encoding="utf-8") as f:
            f.write(content)
        return f"Draft successfully saved to {final_name}"
    except Exception as e:
        return f"Error saving draft: {str(e)}"

# 3. INITIALIZE TOOLS (Using latest langchain-tavily SDK)
# Using 'advanced' depth and including the AI-generated answer for better context
search = TavilySearchResults(
    max_results=5,
    search_depth="advanced",
    include_answer=True,
    include_raw_content=True
)

tools = [search, python_repl_tool, save_draft]

# 4. INITIALIZE LLM & AGENT
# GPT-4o is optimized for tool-calling reliability
llm = ChatOpenAI(model="gpt-4o", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert Research Agent.
    Your goal is to provide comprehensive, data-backed reports.

    Step 1: Use 'tavily_search_results_json' to find facts and current data.
    Step 2: Use 'python_repl_tool' if you need to calculate growth, averages, or process data.
    Step 3: Structure your findings into a professional report.
    Step 4: Use 'save_draft' to save the final document.

    Always explain your calculations and cite your findings briefly.
    """),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# Create the Agent
agent = create_tool_calling_agent(llm, tools, prompt)

# Create the Executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10 # Prevent infinite loops in complex research tasks
)

# 5. EXECUTION EXAMPLE
if __name__ == "__main__":
    query = (
        "Analyze the financial performance of Nvidia (NVDA) and AMD over the last 12 months. "
        "Compare their percentage growth using Python. Save a detailed comparison draft "
        "including a final recommendation on which stock performed better."
    )

    print(f"🚀 Starting Research: {query}\n")

    # In LangChain v0.3+, invoke is the standard entry point
    response = agent_executor.invoke({"input": query})

    print("\n--- Final Agent Response ---")
    print(response["output"])